# Tutorial: `train_gpt.py` — A Complete Walkthrough

## Introduction

This notebook is a **reading guide** for `train_gpt.py`, the reference training script for the [Parameter Golf](https://github.com/karpathy/parameter-golf) competition. Keep the source file open alongside this tutorial; the explanation here follows the code section by section from top to bottom.

---

### What is Parameter Golf?

The goal is to train the best language model that fits inside a **16 MB artifact**, using at most **10 minutes of training time** on **8×H100 GPUs**. The quality metric is **bits-per-byte (BPB)** measured on a held-out slice of the FineWeb dataset. Lower BPB is better — it means the model is a more efficient compressor of English text.

The name "parameter golf" comes from the analogy with code golf: you are trying to get the most out of the fewest parameters, subject to a hard size budget.

---

### What is in this file?

`train_gpt.py` is a **self-contained** submission script. In a single 1,126-line file it contains:

- **Model architecture** — a GPT-style transformer with several modern tweaks
- **Muon optimizer** — a research optimizer for matrix parameters
- **Data loading** — a streaming binary shard reader
- **BPB evaluation** — tokenizer-agnostic quality measurement
- **Post-training quantization** — int8 + zlib compression to meet the 16 MB cap
- **Distributed training** — multi-GPU coordination via `torchrun` / NCCL
- **The training loop** — gradient accumulation, LR scheduling, and a hard wallclock stop

The file is deliberately capped at 1,500 lines to remain approachable for newcomers. This tutorial walks through every one of those lines.

---

### Prerequisites

To follow along comfortably you should be familiar with:

- Python 3.10+
- Basic PyTorch (tensors, `nn.Module`, autograd)
- The idea of a transformer language model (attention, MLP, residual connections)
- Basic concepts of distributed training (ranks, all-reduce)

You do **not** need to understand every mathematical detail up front. The tutorial explains the key ideas as they appear in the code.

---

### Table of Contents

1. [Imports & Setup](#section-1)
2. [Hyperparameters](#section-2)
3. [Muon Optimizer](#section-3)
4. [Tokenizer-Agnostic Evaluation](#section-4)
5. [Post-Training Quantization](#section-5)
6. [Data Loading](#section-6)
7. [Transformer Modules](#section-7)
8. [main() — Distributed & CUDA Setup](#section-8)
9. [Tokenizer & Validation Setup](#section-9)
10. [Model & Optimizer Setup](#section-10)
11. [Compilation Warmup](#section-11)
12. [Main Training Loop](#section-12)
13. [Serialization & Roundtrip Validation](#section-13)
14. [Summary & Levers for Improvement](#summary)

<a id="section-1"></a>

---

## Section 1: Imports & Setup (lines 1–28)

The file opens with two things: a docstring explaining the intent of the script, and all of the imports.

### The opening docstring

```python
"""
The `train_gpt.py` and `train_gpt_mlx.py` scripts are intended as good
launching-off points for new participants, not SOTA configs. We'll accept PRs
that tune, improve, or simplify these scripts without significantly increasing
complexity, but competitive submissions should stay in the /records folder.

Hard stop: To keep readable for newcomers, let's make sure `train_gpt.py` and
`train_gpt_mlx.py` never are longer than 1500 lines.
"""
```

This sets the tone: the script is designed to be **readable**, not maximally optimized. Contestants who want to go further should fork into `/records`.

---

### `from __future__ import annotations`

This line makes all type annotations in the file **lazy strings** evaluated at type-checking time rather than at runtime. The practical effect is that you can write things like:

```python
def foo(x: Tensor | None) -> tuple[Tensor, Tensor]:
```

using the newer `X | Y` union syntax and built-in generics like `tuple[...]` even on Python 3.9, because the annotations are never actually evaluated at runtime. It is a pure ergonomic choice with no runtime cost.

---

### Standard library imports

```python
import copy, glob, io, math, os, random, subprocess, sys, time, uuid, zlib
from pathlib import Path
```

- `copy` — used to snapshot optimizer state before the compilation warmup
- `glob` — finds shard files matching patterns like `fineweb_train_*.bin`
- `io` — `BytesIO` for in-memory serialization during quantization export
- `math` — `math.log(2.0)` for the nats-to-bits conversion in BPB
- `os` — reading environment variables for all hyperparameters
- `random` — seeding Python's RNG for reproducibility
- `subprocess` — running `nvidia-smi` to log GPU info at startup
- `sys` — logging the Python version
- `time` — `time.perf_counter()` for the wallclock timer
- `uuid` — generating a unique `run_id` if one is not provided
- `zlib` — compressing the quantized model artifact
- `Path` — ergonomic file path handling for shard loading

---

### External imports

```python
import numpy as np
import sentencepiece as spm
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import Tensor, nn
from torch.nn.parallel import DistributedDataParallel as DDP
```

- **`numpy`** — reading binary shard files (faster than pure Python for uint16 arrays) and seeding its own RNG
- **`sentencepiece`** — the tokenizer library. SentencePiece supports BPE and unigram models and is compact to deploy
- **`torch`** — the core training framework
- **`torch.distributed`** (aliased `dist`) — all inter-GPU communication: `all_reduce`, `barrier`, process group init/destroy
- **`torch.nn.functional`** (aliased `F`) — stateless operations: `F.linear`, `F.rms_norm`, `F.cross_entropy`, `F.scaled_dot_product_attention`
- **`Tensor, nn`** — imported directly from `torch` to reduce typing throughout the file
- **`DistributedDataParallel`** (aliased `DDP`) — the PyTorch wrapper that handles gradient synchronization across GPUs

Notice there are **no imports from a separate `model.py`, `config.py`, or `utils.py`**. Everything lives in this one file, which is a deliberate design choice for submission portability.

<a id="section-2"></a>

---

## Section 2: Hyperparameters (lines 30–87)

```python
class Hyperparameters:
    data_path = os.environ.get("DATA_PATH", "./data/datasets/fineweb10B_sp1024")
    ...
```

### Design pattern: environment variables with class-level defaults

All configuration is expressed as **class-level attributes** on `Hyperparameters`. There is no `__init__` method. This means you instantiate it with `args = Hyperparameters()` and get an object whose attributes are already populated from the environment.

The `os.environ.get("KEY", default)` pattern means:
- If you set `KEY=value` in your shell before running, that value is used.
- Otherwise the default is used.
- No config files, no argparse, no YAML. Everything is overridable from the shell without touching the code.

```bash
TRAIN_SEQ_LEN=2048 NUM_LAYERS=12 torchrun --nproc_per_node=8 train_gpt.py
```

This makes the script trivially forkable — change a hyperparameter by setting an env var.

---

### Group 1: Data paths

```python
data_path    = os.environ.get("DATA_PATH", "./data/datasets/fineweb10B_sp1024")
train_files  = os.path.join(data_path, "fineweb_train_*.bin")
val_files    = os.path.join(data_path, "fineweb_val_*.bin")
tokenizer_path = os.environ.get("TOKENIZER_PATH", "./data/tokenizers/fineweb_1024_bpe.model")
run_id       = os.environ.get("RUN_ID", str(uuid.uuid4()))
seed         = int(os.environ.get("SEED", 1337))
```

- `train_files` and `val_files` are **glob patterns**, not individual filenames. The data is stored in multiple binary shards that are discovered at runtime.
- `tokenizer_path` points to a `.model` file — the SentencePiece format.
- `run_id` is a UUID used to name the log file. If you run multiple experiments, each gets its own log.
- `seed` controls all RNG sources. Using the same seed on the same hardware produces bit-identical results.

---

### Group 2: Validation cadence

```python
val_batch_size  = int(os.environ.get("VAL_BATCH_SIZE", 524_288))   # ~512K tokens
val_loss_every  = int(os.environ.get("VAL_LOSS_EVERY", 1000))       # validate every N steps
train_log_every = int(os.environ.get("TRAIN_LOG_EVERY", 200))       # log training loss every N steps
```

`val_batch_size` of 524,288 tokens (512K) means each validation pass processes roughly half a million tokens. With `train_seq_len=1024` that is 512 sequences. The validation code always uses the **full** validation split regardless of this number; `val_batch_size` only controls the batch size used per micro-step, which affects memory but not which data is evaluated.

---

### Group 3: Training length

```python
iterations          = 20000   # maximum number of training steps
warmdown_iters      = 1200    # LR linearly decays to 0 over the last N steps
warmup_steps        = 20      # steps of torch.compile warmup (throwaway)
train_batch_tokens  = 524_288 # total tokens per optimizer step (across all GPUs)
train_seq_len       = 1024    # sequence length for training
max_wallclock_seconds = 600.0 # hard 10-minute cap
```

The most important number here is `train_batch_tokens = 524,288`. This is the **global batch size** in tokens, held constant regardless of how many GPUs you use. With 8 GPUs and 1 gradient accumulation step, each GPU processes `524288 / 8 = 65536` tokens per step. With 1 GPU and 8 accumulation steps, that single GPU processes `524288 / 8 = 65536` tokens per micro-step but still 524,288 tokens total before each optimizer update.

`max_wallclock_seconds = 600` is the 10-minute competition cap. The script enforces this programmatically and gracefully stops training right as the timer runs out.

---

### Group 4: Model shape

```python
vocab_size   = 1024   # tiny vocabulary!
num_layers   = 9
num_kv_heads = 4      # fewer KV heads than query heads (GQA)
model_dim    = 512
num_heads    = 8
mlp_mult     = 2      # hidden MLP size = 2 × model_dim
tie_embeddings = True # reuse embedding matrix as output projection
rope_base    = 10000.0
logit_softcap = 30.0
```

**The tiny vocabulary is the single most important design decision here.** The embedding table is `vocab_size × model_dim` parameters, which with the defaults is `1024 × 512 = 524,288` parameters. Each parameter is stored in fp32 at 4 bytes, so the embedding table is about 2 MB by itself.

Compare this to GPT-2's vocabulary of 50,257: at the same `model_dim=512`, that would be `50257 × 512 × 4 = ~103 MB` — larger than the entire 16 MB budget. A tiny vocabulary is the **necessary prerequisite** for fitting any real transformer into 16 MB.

The trade-off is that a vocabulary of 1024 produces much longer token sequences for the same text (more tokens per byte), which means the model has to process more tokens to see the same amount of information. This tension between vocabulary size and sequence length is a key design lever for participants.

`tie_embeddings = True` means the same 524K-parameter matrix is used both to map token IDs to vectors at the input and to project the final hidden state back to logits at the output. This halves the embedding parameter cost.

---

### Group 5: Optimizer hyperparameters

```python
embed_lr        = 0.6    # Adam LR for token embedding (when NOT tied)
head_lr         = 0.008  # Adam LR for output head (when NOT tied)
tied_embed_lr   = 0.05   # Adam LR for token embedding (when tied)
matrix_lr       = 0.04   # Muon LR for weight matrices
scalar_lr       = 0.04   # Adam LR for 1D parameters (scales, biases, etc.)

muon_momentum              = 0.95   # steady-state momentum for Muon
muon_momentum_warmup_start = 0.85   # momentum starts here and ramps up
muon_momentum_warmup_steps = 500    # number of steps to ramp

beta1 = 0.9
beta2 = 0.95
grad_clip_norm = 0.0  # 0 means disabled
```

The script uses **four separate optimizers** with different learning rates:

1. **`optimizer_tok`** — Adam for the token embedding, LR 0.05 (tied) or 0.6 (untied)
2. **`optimizer_head`** — Adam for the output head, LR 0.008 (only exists when not tied)
3. **`optimizer_muon`** — Muon for all 2D weight matrices in transformer blocks, LR 0.04
4. **`optimizer_scalar`** — Adam for 1D parameters (scales, `resid_mix`, `q_gain`, skip weights), LR 0.04

Why different learning rates? Embeddings and the output head operate on a different scale than the internal transformer weights. The Muon optimizer also produces updates with a different effective scale than Adam, because it orthogonalizes gradients. Each group's LR was tuned separately.

Note that `grad_clip_norm = 0.0` means gradient clipping is **disabled** by default. The combination of Muon (which normalizes matrix gradients) and the small per-dimension scale parameters (`attn_scale`, `mlp_scale`) provides enough implicit regularization that clipping is not needed.

<a id="section-3"></a>

---

## Section 3: Muon Optimizer (lines 89–168)

Muon is a research optimizer created for the [modded-nanogpt](https://github.com/KellerJordan/modded-nanogpt) project. The idea is to replace Adam's coordinate-wise update for matrix-shaped parameters with an **orthogonalized** update — one that lives on the Stiefel manifold of matrices with orthonormal rows (or columns). Empirically, this trains transformer weight matrices faster than Adam for the same step budget.

For a deeper background, see [Keller Jordan's blog post on Muon](https://kellerjordan.github.io/posts/muon/).

---

### `zeropower_via_newtonschulz5()` (lines 96–109)

```python
def zeropower_via_newtonschulz5(G: Tensor, steps: int = 10, eps: float = 1e-7) -> Tensor:
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X /= X.norm() + eps
    transposed = G.size(0) > G.size(1)
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    return X.T if transposed else X
```

This function takes a 2D gradient matrix `G` and returns its **orthogonal factor** — the matrix `U @ V.T` from the singular value decomposition `G = U @ diag(s) @ V.T`. In other words, it strips out the singular values and keeps only the directions.

**Why this is the right thing to do for an optimizer:**

Adam scales each coordinate's gradient by the inverse of its historical RMS — it normalizes per-scalar. Muon normalizes per-matrix: instead of scaling individual entries, it finds the update direction that has the same "angular" effect as the gradient but with all singular values set to 1. This is a much more aggressive normalization, and it empirically leads to faster convergence on the weight matrices of transformers.

**The Newton-Schulz iteration:**

The iteration `X = a*X + (b*A + c*A@A) @ X` where `A = X @ X.T` is a matrix polynomial that converges to the orthogonal factor of `X`. Think of it as a fast, differentiable alternative to computing a full SVD. The coefficients `(3.4445, -4.7750, 2.0315)` are numerically optimized to converge in about 5 iterations rather than the ~30 iterations that naive choices would require.

**The transpose trick:** The iteration works more efficiently when the matrix has more columns than rows (wide matrix). If `G` is tall (more rows than columns), the code transposes it before iterating and transposes back at the end. Both forms produce the same result.

**Precision:** The computation is done in `bfloat16` for speed. The result is good enough — small errors in the orthogonalization do not significantly impact training quality.

---

### `Muon` class (lines 112–168)

The `Muon` class is a standard PyTorch optimizer (`torch.optim.Optimizer`). Its `step()` method is called once per training step and updates all parameters in its parameter groups.

#### Constructor

```python
def __init__(self, params, lr: float, momentum: float, backend_steps: int, nesterov: bool = True):
```

`backend_steps` controls how many Newton-Schulz iterations to run (default 5). `nesterov=True` applies Nesterov momentum, which "looks ahead" in the gradient direction and tends to reduce oscillations.

#### The `step()` method: distributed work sharding

The most interesting part of Muon is how it handles multi-GPU training. The Newton-Schulz iteration is expensive (it involves matrix multiplications), and we do not want every GPU doing all the work redundantly:

```python
for i, p in enumerate(params):
    if i % world_size == rank and p.grad is not None:
        ...  # this rank only processes its assigned parameters
```

With 8 GPUs (world_size=8) and, say, 72 matrix parameters, each GPU processes 9 of the 72 matrices. The work is divided by index: rank 0 handles parameters 0, 8, 16, ...; rank 1 handles 1, 9, 17, ...; and so on.

#### Per-parameter update computation

For each assigned parameter:

1. **Accumulate momentum:**
   ```python
   buf.mul_(momentum).add_(g)
   ```
   This is standard exponential moving average of gradients (the "heavy ball" step).

2. **Nesterov look-ahead** (when enabled):
   ```python
   g = g.add(buf, alpha=momentum)
   ```
   Instead of using the buffered gradient directly, Nesterov momentum adds a fraction of the current buffer to the raw gradient. This "looks ahead" to where the momentum will carry you.

3. **Orthogonalize:**
   ```python
   g = zeropower_via_newtonschulz5(g, steps=backend_steps)
   ```
   This is the Muon-specific step. The gradient is replaced by its orthogonal factor.

4. **Scale correction:**
   ```python
   g *= max(1, g.size(0) / g.size(1)) ** 0.5
   ```
   After orthogonalization, all singular values are 1. For a matrix with more rows than columns (e.g., `512×256`), the norm of the orthogonalized gradient is `sqrt(cols)`, while for a square matrix it is `sqrt(rows) = sqrt(cols)`. For a tall matrix, this correction factor ensures the update has the same effective magnitude as it would for a square matrix.

#### Aggregation via all_reduce

After each GPU has computed its subset of orthogonalized updates:

```python
updates_flat = torch.zeros(total_params, ...)
# ... fill in assigned positions ...
dist.all_reduce(updates_flat, op=dist.ReduceOp.SUM)
```

All parameter updates are packed into a single flat buffer. Since each GPU wrote to disjoint positions (parameter `i` was only processed by rank `i % world_size`), summing across all ranks via `all_reduce(SUM)` gives every GPU the complete set of updates. This is a single network communication round-trip regardless of the number of parameters.

Finally:
```python
p.add_(g, alpha=-lr)
```
The parameter is updated in-place using the lr-scaled orthogonalized gradient.

#### Why Muon over Adam for matrices?

Adam adapts per-scalar. For a `512×512` weight matrix, Adam maintains `512×512` momentum and variance estimates and applies `512×512` independent scalar updates. Muon treats the entire matrix as a single geometric object and updates it as an orthogonal transformation — it moves the matrix on the Stiefel manifold. This tends to distribute the learning signal more evenly across all dimensions of the weight matrix, especially early in training when some directions may have much larger gradients than others.

<a id="section-4"></a>

---

## Section 4: Tokenizer-Agnostic Evaluation (lines 170–278)

This section solves a subtle fairness problem in the competition.

**The problem:** If one participant uses a vocabulary of 1024 tokens and another uses 8192 tokens, their models will produce different numbers of tokens for the same text. A model with a larger vocabulary produces fewer, richer tokens; a model with a smaller vocabulary produces more, simpler tokens. If you measure model quality as "cross-entropy loss per token", these are not comparable. A model with a smaller vocabulary can achieve a lower per-token loss simply by having longer token sequences, not because it is actually compressing language better.

**The solution:** Measure **bits-per-byte (BPB)** instead of bits-per-token. BPB counts how many bits of information the model assigns (on average) to each *byte* of the original text. This normalizes out the tokenizer's compression factor and makes different tokenizations directly comparable.

The section implements the machinery to convert per-token loss into per-byte loss.

---

### `build_sentencepiece_luts()` (lines 180–204)

```python
def build_sentencepiece_luts(
    sp: spm.SentencePieceProcessor, vocab_size: int, device: torch.device
) -> tuple[Tensor, Tensor, Tensor]:
```

This function builds three **lookup tables** (LUTs), indexed by token ID, that tell us how many bytes each token represents. The tables are returned as GPU tensors so they can be used directly in the validation loop.

**Table 1: `base_bytes`** (dtype `int16`)
For each token ID, how many UTF-8 bytes does the decoded token occupy? This is straightforward for regular tokens: the string `"hello"` has 5 bytes. For byte-fallback tokens (single raw bytes), the answer is always 1.

**Table 2: `has_leading_space`** (dtype `bool`)
SentencePiece uses the special symbol `▁` (a Unicode lower one-eighth block, U+2581) as a word-boundary marker. A token like `▁hello` represents a space followed by "hello". The `▁` itself is 3 UTF-8 bytes but it encodes the space that precedes the word, not the word itself. This boolean table flags which tokens start with `▁`.

**Table 3: `is_boundary_token`** (dtype `bool`)
Control tokens (BOS, EOS), unknown tokens, and unused vocabulary slots are "boundary tokens" — they don't represent real text content. This is used to handle the leading-space logic at sentence boundaries correctly: we should not count a leading space at the very start of a text where there is no previous word.

**Token ID iteration:**
```python
for token_id in range(sp_vocab_size):
    if sp.is_control(token_id) or sp.is_unknown(token_id) or sp.is_unused(token_id):
        continue
    is_boundary_token_np[token_id] = False
    if sp.is_byte(token_id):
        base_bytes_np[token_id] = 1
        continue
    piece = sp.id_to_piece(token_id)
    if piece.startswith("▁"):
        has_leading_space_np[token_id] = True
        piece = piece[1:]
    base_bytes_np[token_id] = len(piece.encode("utf-8"))
```

Notice that `is_boundary_token_np` is **initialized to all `True`** and then flipped to `False` for real tokens. This means any token ID outside the actual vocabulary (which can happen if `vocab_size > sp_vocab_size`) is treated as a boundary token by default — a safe fallback.

---

### `load_validation_tokens()` (lines 207–216)

```python
tokens = torch.cat([load_data_shard(file) for file in files]).contiguous()
usable = ((tokens.numel() - 1) // seq_len) * seq_len
return tokens[: usable + 1]
```

This loads all validation shard files and truncates the result to be exactly divisible by `seq_len`. The `+1` at the end is critical: to create input/target pairs for language modeling, we need `N+1` tokens to form `N` (input, target) pairs where the target is shifted by one position.

---

### `eval_val()` (lines 219–278)

This is the full validation loop. It runs the model over the entire validation set and returns both the standard cross-entropy loss and the BPB score.

#### Data distribution across ranks

```python
seq_start = (total_seqs * rank) // world_size
seq_end   = (total_seqs * (rank + 1)) // world_size
```

The validation sequences are divided equally across all GPUs. Rank 0 handles the first `1/world_size` of sequences, rank 1 handles the next `1/world_size`, and so on. This means validation is also parallelized — with 8 GPUs, validation runs 8× faster.

#### The inference loop

```python
model.eval()
with torch.inference_mode():
    for batch_seq_start in range(seq_start, seq_end, local_batch_seqs):
        ...
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
            batch_loss = model(x, y).detach()
```

`torch.inference_mode()` is a more aggressive version of `torch.no_grad()` — it disables gradient tracking AND the version counter on tensors, resulting in slightly lower memory usage and faster execution.

`torch.autocast` runs the forward pass in `bfloat16`. The loss is computed in `float32` (cross-entropy internally promotes to float32) then immediately detached from the computation graph.

#### The byte counting (lines 263–267)

This is the core of the BPB calculation:

```python
prev_ids = x.reshape(-1)         # the input tokens (positions 0..N-1)
tgt_ids  = y.reshape(-1)         # the target tokens (positions 1..N)
token_bytes = base_bytes_lut[tgt_ids].to(dtype=torch.int16)
token_bytes += (has_leading_space_lut[tgt_ids] & ~is_boundary_token_lut[prev_ids]).to(dtype=torch.int16)
val_byte_count += token_bytes.to(torch.float64).sum()
```

For each target token:
1. Start with its base byte count (from `base_bytes_lut`).
2. Add 1 if it has a leading space **AND** the token that precedes it is not a boundary token.

The second condition handles a subtle edge case: at the very start of a sequence (or after an EOS token), the previous token is a boundary token, so we should **not** count the leading space as a real byte — it is an artifact of the sentence start, not a space in the original text.

#### Converting loss to BPB

```python
bits_per_token  = val_loss.item() / math.log(2.0)
tokens_per_byte = val_token_count.item() / val_byte_count.item()
bpb             = bits_per_token * tokens_per_byte
```

The cross-entropy loss is in **nats** (natural log units). Dividing by `ln(2) ≈ 0.693` converts to **bits**. Then multiplying by `tokens_per_byte` (how many tokens are needed per byte of original text) gives bits per byte.

A BPB of 1.0 would mean the model uses exactly 1 bit per byte — perfect compression to the Shannon limit. Typical language models for English achieve something in the range of 0.9–1.2 BPB depending on model size.

#### All-reduce for global metrics

```python
dist.all_reduce(val_loss_sum,   op=dist.ReduceOp.SUM)
dist.all_reduce(val_token_count, op=dist.ReduceOp.SUM)
dist.all_reduce(val_byte_count,  op=dist.ReduceOp.SUM)
```

Each rank has accumulated sums over its slice of the validation set. Summing across all ranks gives the global totals, from which the global metrics are computed.

<a id="section-5"></a>

---

## Section 5: Post-Training Quantization (lines 280–422)

The competition requires the entire submission — model weights plus the Python source file — to be under 16 MB. After training in `bfloat16` and `float32`, the raw model state dict is far too large. This section implements the pipeline that compresses it down to the required size.

The strategy has three steps:
1. **Int8 quantization** — represent each weight as an 8-bit integer plus a scale factor
2. **In-memory serialization** — pack everything into a `torch.save` blob
3. **Zlib compression** — apply general-purpose compression to the serialized blob

Together, these typically achieve a **4–6× size reduction** compared to the raw `bfloat16` model.

---

### Configuration constants (lines 288–308)

```python
CONTROL_TENSOR_NAME_PATTERNS = ("attn_scale", "mlp_scale", "resid_mix", "q_gain",
                                 "skip_weight", ...)
INT8_KEEP_FLOAT_MAX_NUMEL = 65_536
INT8_KEEP_FLOAT_STORE_DTYPE = torch.float16
INT8_PER_ROW_SCALE_DTYPE = torch.float16
INT8_CLIP_PERCENTILE = 99.99984
```

**`CONTROL_TENSOR_NAME_PATTERNS`:** These are the names of small learnable scaling parameters — `attn_scale`, `mlp_scale`, `resid_mix`, `q_gain`, and `skip_weight`. These tensors are numerically sensitive (they control the scale of residual additions) and are also very small, so they are kept in full `float32` precision rather than quantized. They are also exempt from the "convert to fp16" downcast applied to other small tensors.

**`INT8_KEEP_FLOAT_MAX_NUMEL = 65,536`:** Any tensor with ≤ 65,536 elements is kept as `float16` rather than quantized to int8. The quantization format stores a scale factor alongside the int8 data. For small tensors, this overhead is disproportionately large — a tensor with 64 elements would need 64 int8 values (64 bytes) plus 64 fp16 scales (128 bytes for per-row) totaling 192 bytes, versus just 64 fp16 values at 128 bytes. The threshold at 65,536 elements is where int8 starts to be cost-effective.

**`INT8_CLIP_PERCENTILE = 99.99984`:** Before quantizing, the code clips the top and bottom 0.00016% of values (in absolute magnitude). Neural network weight matrices almost always have a small number of "outlier" values that are much larger than the typical value. If you scale the int8 range to cover these outliers, you waste most of the 256 available quantization levels on a handful of values, degrading the precision of the typical values. By clipping at the 99.99984th percentile, these outliers are clamped to the clip threshold and the remaining 99.99984% of values get the full int8 resolution.

---

### `quantize_float_tensor()` (lines 321–340)

This function quantizes a single tensor to int8. It handles two cases differently.

**2D tensors (weight matrices) — per-row quantization:**

```python
clip_abs = torch.quantile(t32.abs(), INT8_CLIP_Q, dim=1)  # one clip value per row
clipped  = torch.maximum(torch.minimum(t32, clip_abs[:, None]), -clip_abs[:, None])
scale    = (clip_abs / 127.0).clamp_min(1.0 / 127.0)      # one scale per row
q = torch.clamp(torch.round(clipped / scale[:, None]), -127, 127).to(torch.int8)
```

Per-row quantization is used because different rows of a weight matrix (corresponding to different output neurons) often have very different value ranges. Per-row scaling allows each row to use the full int8 range independently. The int8 range is `[-127, 127]` (not `-128` because symmetric quantization makes dequantization simpler: just multiply by scale). The scale is stored as `float16`, one value per row.

**1D/0D tensors (vectors, scalars) — per-tensor quantization:**

```python
clip_abs = float(torch.quantile(t32.abs().flatten(), INT8_CLIP_Q).item())
scale    = torch.tensor(clip_abs / 127.0 ..., dtype=torch.float32)
q = torch.clamp(torch.round(torch.clamp(t32, -clip_abs, clip_abs) / scale), -127, 127).to(torch.int8)
```

For a bias vector or scalar, per-row doesn't apply. A single scale for the entire tensor is used instead.

---

### `quantize_state_dict_int8()` (lines 342–399)

This function processes the entire model state dict and returns a structured dictionary containing:

- **`quantized`** — the int8 tensors for large float tensors
- **`scales`** — the corresponding scale factors (float16 per-row for matrices, float32 scalar for vectors)
- **`dtypes`** — the original dtype of each quantized tensor (so dequantization can cast back)
- **`passthrough`** — tensors kept at their original precision (small tensors, non-float tensors)
- **`passthrough_orig_dtypes`** — for passthrough tensors that were downcast from bf16/fp32 to fp16, the original dtype is stored here so dequantization can restore it
- **`qmeta`** — metadata about the quantization scheme (per-row vs. per-tensor)

The decision tree for each tensor:

1. **Not a float type** (e.g., int64 token IDs): pass through unchanged.
2. **Float with ≤ 65,536 elements**: keep as fp16 (or fp32 for control tensors), no quantization.
3. **Float with > 65,536 elements**: quantize to int8 with per-row or per-tensor scale.

---

### `dequantize_state_dict_int8()` (lines 401–422)

This reverses the quantization:

```python
for name, q in obj["quantized"].items():
    dtype = getattr(torch, obj["dtypes"][name])
    s = obj["scales"][name]
    if qmeta.get(name, {}).get("scheme") == "per_row" or s.ndim > 0:
        s = s.to(dtype=torch.float32)
        out[name] = (q.float() * s.view(q.shape[0], *([1] * (q.ndim - 1)))).to(dtype=dtype)
    else:
        out[name] = (q.float() * float(s.item())).to(dtype=dtype)
```

For per-row quantization, the scale `s` has shape `[rows]`. We need to broadcast it across the column dimension: `s.view(rows, 1)` allows `s * q` to broadcast correctly. The reconstruction formula is just `dequant = int8_value * scale`.

Passthrough tensors get their original dtype restored from the saved metadata:

```python
orig_dtype = passthrough_orig_dtypes.get(name)
if isinstance(orig_dtype, str):
    out_t = out_t.to(dtype=getattr(torch, orig_dtype))
```

The final result is a state dict in the model's original dtypes, ready to be loaded back with `model.load_state_dict()`.

---

### Why zlib on top of int8?

Even after quantization, the int8 weights are not random noise — they have structure (nearby weights tend to be correlated, many values cluster near zero). Zlib (DEFLATE compression) exploits this structure and typically achieves another 2–3× compression on top of int8. Maximum compression level (`level=9`) is used since compression time is a one-time cost paid at the end of training, not during the hot loop.

<a id="section-6"></a>

---

## Section 6: Data Loading (lines 425–494)

The training data for this competition is FineWeb, a large web-crawl dataset. It is pre-tokenized and stored as binary files called "shards". This section implements the machinery to read those shards efficiently.

---

### `load_data_shard()` (lines 429–443)

```python
def load_data_shard(file: Path) -> Tensor:
    header_bytes = 256 * np.dtype("<i4").itemsize   # 256 × 4 = 1024 bytes
    token_bytes  = np.dtype("<u2").itemsize          # 2 bytes per uint16 token

    header = np.fromfile(file, dtype="<i4", count=256)
    if header.size != 256 or int(header[0]) != 20240520 or int(header[1]) != 1:
        raise ValueError(...)
    num_tokens = int(header[2])
    ...
    tokens_np = np.fromfile(file, dtype="<u2", count=num_tokens, offset=header_bytes)
    return torch.from_numpy(tokens_np.astype(np.uint16, copy=False))
```

**Binary format:** Each shard file has a fixed 256-int32 header (1024 bytes total) followed by the raw token data. The header fields are:

- `header[0]` — magic number `20240520` (the date the format was created)
- `header[1]` — version number `1`
- `header[2]` — number of tokens in this shard

**Token dtype:** Tokens are stored as **unsigned 16-bit integers** (`uint16`). This supports vocabularies up to 65,535 tokens — far more than the competition's default vocabulary of 1024. With 2 bytes per token, a shard of 10M tokens occupies 20 MB on disk.

**Validation:** The function checks the magic number, version, file size, and actual number of tokens read. This catches corrupted or truncated shard files early and clearly, rather than producing mysterious downstream errors.

**Why numpy for file I/O?** `np.fromfile` is significantly faster than reading the file in Python and constructing a tensor element-by-element. It maps directly to a C `fread()` call and avoids Python overhead. The result is immediately wrapped in a PyTorch tensor with `torch.from_numpy`.

---

### `TokenStream` (lines 446–474)

```python
class TokenStream:
    def __init__(self, pattern: str):
        self.files = [Path(p) for p in sorted(glob.glob(pattern))]
        self.file_idx = 0
        self.tokens = load_data_shard(self.files[0])
        self.pos = 0
```

`TokenStream` provides a simple sequential cursor over all shard files. It reads one shard at a time into memory. When the current shard is exhausted, it advances to the next one. When all shards are exhausted, it wraps back to the first shard.

The `take(n)` method returns exactly `n` tokens from the stream, spanning shard boundaries if needed:

```python
def take(self, n: int) -> Tensor:
    chunks: list[Tensor] = []
    remaining = n
    while remaining > 0:
        avail = self.tokens.numel() - self.pos
        if avail <= 0:
            self._advance_file()
            continue
        k = min(remaining, avail)
        chunks.append(self.tokens[self.pos : self.pos + k])
        self.pos += k
        remaining -= k
    return chunks[0] if len(chunks) == 1 else torch.cat(chunks)
```

This is deliberately simple. There is:
- No shuffling (data is seen in fixed shard order)
- No random access or seeking
- No background prefetching or worker threads
- No data augmentation

For a 10-minute training run, this simplicity is the right trade-off. The H100s can process data faster than a single-threaded reader can serve it, but the bottleneck in this competition is computation, not I/O — the entire dataset is 10B tokens and the training run only sees ~10B tokens total in 20K steps, so we go through the data roughly once.

---

### `DistributedTokenLoader` (lines 477–494)

```python
class DistributedTokenLoader:
    def next_batch(self, global_tokens: int, seq_len: int, grad_accum_steps: int):
        local_tokens = global_tokens // (self.world_size * grad_accum_steps)
        per_rank_span = local_tokens + 1
        chunk = self.stream.take(per_rank_span * self.world_size)
        start = self.rank * per_rank_span
        local = chunk[start : start + per_rank_span].to(dtype=torch.int64)
        x = local[:-1].reshape(-1, seq_len)
        y = local[1:].reshape(-1, seq_len)
        return x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
```

This class wraps `TokenStream` for multi-GPU use. Each call to `next_batch()` provides one micro-step's worth of data for the calling rank.

**How the math works out:**

`global_tokens` is `train_batch_tokens = 524,288`. Each rank processes `global_tokens / (world_size × grad_accum_steps)` tokens per micro-step. With 8 GPUs and 1 accumulation step, that is `524288 / (8 × 1) = 65,536` tokens per micro-step per GPU.

`per_rank_span = local_tokens + 1 = 65,537`. The "+1" is needed because input-target pairs are formed by offsetting by 1: the input is tokens `[0, local_tokens)` and the target is tokens `[1, local_tokens + 1)`. So we need to read `local_tokens + 1` tokens to form `local_tokens` pairs.

**Rank-based slicing:** All ranks call `take()` on the same `TokenStream` in the same order — the stream is advanced collectively. Then each rank slices out its own disjoint span:

```
rank 0 gets: chunk[0 : per_rank_span]
rank 1 gets: chunk[per_rank_span : 2 * per_rank_span]
...
rank N gets: chunk[N * per_rank_span : (N+1) * per_rank_span]
```

This ensures every rank processes a different portion of the data at each step, and together the ranks cover the full global batch.

**Final reshape:** After slicing, `local` has `per_rank_span` tokens. Taking `local[:-1]` (all but last) as `x` and `local[1:]` (all but first) as `y`, then reshaping to `(-1, seq_len)`, gives a batch of `(batch_size, seq_len)` input/target tensor pairs. With `local_tokens = 65,536` and `seq_len = 1024`, `batch_size = 65536 / 1024 = 64`.

**`non_blocking=True`:** The `.to(self.device)` transfer is non-blocking, meaning the CPU can continue executing while the H2D copy happens asynchronously on a separate CUDA stream. This hides data transfer latency behind computation from the previous step.

<a id="section-7"></a>

---

## Section 7: Transformer Modules (lines 496–724)

This section defines the neural network architecture. The model is a GPT-style autoregressive transformer with several modifications relative to the original GPT-2 design. Each module is covered in the order it appears in the file.

---

### `RMSNorm` (lines 500–506)

```python
class RMSNorm(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return F.rms_norm(x, (x.size(-1),), eps=self.eps)
```

**Root Mean Square Normalization** divides each vector by its root mean square:

```
RMSNorm(x) = x / sqrt(mean(x²) + eps)
```

Compare this to LayerNorm, which both centers and normalizes:

```
LayerNorm(x) = (x - mean(x)) / sqrt(var(x) + eps) * gamma + beta
```

RMSNorm omits the mean subtraction (centering) and the learnable `gamma` and `beta` parameters. The intuition is that centering adds a constant cost (compute + memory) for questionable benefit — transformers appear to be able to represent centered distributions via the residual stream without explicit centering in the norm. In practice, RMSNorm matches LayerNorm quality at lower computational cost, and it is now the default in Llama, Mistral, Gemma, and most other modern open-source models.

Note: there is no learnable scale (`gamma`) or bias (`beta`) in this implementation. Many implementations add a learnable scale, but this script omits it for simplicity and parameter count.

---

### `CastedLinear` (lines 509–513)

```python
class CastedLinear(nn.Linear):
    def forward(self, x: Tensor) -> Tensor:
        bias = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, self.weight.to(x.dtype), bias)
```

This is a `nn.Linear` that casts its weight to match the input dtype at forward time. The weight is stored in `float32` (for optimizer numerical stability) but the actual matmul executes in `bfloat16` (for speed, because H100s have tensor cores that run bf16 at 2× the throughput of fp32).

The cast `self.weight.to(x.dtype)` produces a temporary bf16 copy of the weight for the matmul, then discards it. The stored weight remains in fp32. When the Adam optimizer updates the weight, it sees the fp32 values and applies precise updates (small LR × gradient is numerically well-represented in fp32 but could underflow in bf16).

---

### `restore_low_dim_params_to_fp32()` (lines 516–521)

```python
def restore_low_dim_params_to_fp32(module: nn.Module) -> None:
    with torch.no_grad():
        for name, param in module.named_parameters():
            if (param.ndim < 2 or any(pattern in name for pattern in CONTROL_TENSOR_NAME_PATTERNS)) \
               and param.dtype != torch.float32:
                param.data = param.data.float()
```

After calling `model.bfloat16()` to move the entire model to bf16, this function finds all parameters that are:
- Less than 2-dimensional (vectors, scalars), OR
- Named as control tensors (`attn_scale`, `mlp_scale`, `resid_mix`, `q_gain`, `skip_weight`)

...and converts them back to `float32`.

**Why these specifically?** These are parameters with few elements (no speed impact to keep in fp32) but significant numerical sensitivity. For example, `resid_mix` is a `[2, model_dim]` parameter that controls how much the block mixes the current hidden state with the original embedding. If this is in bf16, small differences in scale can cause the mixing weights to lose precision and behave erratically early in training when they are near zero.

---

### `Rotary` and `apply_rotary_emb()` (lines 524–552)

**RoPE (Rotary Position Embeddings)** encodes position information by rotating query and key vectors in complex space. Unlike learned position embeddings (which are a lookup table of `seq_len × dim` parameters) or sinusoidal embeddings (which add position to the input), RoPE modifies the query and key vectors directly. The key property is that the dot product `q_i · k_j` depends only on the relative position `i - j`, which gives the model useful inductive bias for sequence-length generalization.

**`Rotary.__init__`:** Precomputes the inverse frequencies:

```python
inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
```

For `head_dim=64` and `base=10000`, this produces 32 frequencies ranging from `1/10000^0 = 1.0` down to `1/10000^(62/64) ≈ 0.00014`. These are the rotation rates for each dimension pair — lower-index dimensions rotate faster (high frequency, captures short-range position) and higher-index dimensions rotate slower (low frequency, captures long-range position).

**`Rotary.forward`:** Builds the cosine and sine tables for a given sequence length:

```python
t     = torch.arange(seq_len, ...)            # positions 0, 1, 2, ..., seq_len-1
freqs = torch.outer(t, inv_freq)               # shape: [seq_len, dim/2]
cos   = freqs.cos()[None, None, :, :]          # shape: [1, 1, seq_len, dim/2]
sin   = freqs.sin()[None, None, :, :]          # shape: [1, 1, seq_len, dim/2]
```

The result is cached keyed on sequence length and device. The `[1, 1, seq_len, dim/2]` shape broadcasts across batch and heads in the attention forward pass.

**`apply_rotary_emb`:**

```python
def apply_rotary_emb(x: Tensor, cos: Tensor, sin: Tensor) -> Tensor:
    half = x.size(-1) // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((x1 * cos + x2 * sin, x1 * (-sin) + x2 * cos), dim=-1)
```

This splits the head dimension in half and applies a 2D rotation to each pair: `(x1, x2) → (x1·cos + x2·sin, -x1·sin + x2·cos)`. This is exactly a complex number multiplication: treating `(x1 + i·x2)` as a complex number and multiplying by `e^(i·freq·pos) = cos + i·sin`.

The result is a vector where each dimension pair encodes the original value rotated by an angle proportional to the position and the dimension's frequency. When two such vectors are dotted (as in attention), the rotation angles subtract, leaving only the relative angle — which encodes the relative position.

---

### `CausalSelfAttention` (lines 555–603)

This is the attention module. Several noteworthy design choices:

#### Grouped Query Attention (GQA)

```python
self.c_q  = CastedLinear(dim, dim,    bias=False)   # Q: dim → dim
self.c_k  = CastedLinear(dim, kv_dim, bias=False)   # K: dim → kv_dim
self.c_v  = CastedLinear(dim, kv_dim, bias=False)   # V: dim → kv_dim
```

With `num_heads=8` and `num_kv_heads=4`, `kv_dim = 4 × head_dim = 4 × 64 = 256`. So Q has dimension 512 but K and V only have dimension 256. This means:
- Q weight: `512 × 512 = 262,144` parameters
- K weight: `512 × 256 = 131,072` parameters
- V weight: `512 × 256 = 131,072` parameters

GQA halves the KV parameter count (and KV cache memory for inference) with minimal quality loss. The 8 query heads are grouped into 4 groups, and each group shares a single K and V head. PyTorch's `F.scaled_dot_product_attention` natively supports GQA when `enable_gqa=True`.

#### QK-norm

```python
q = F.rms_norm(q, (q.size(-1),))
k = F.rms_norm(k, (k.size(-1),))
```

After projecting to Q and K, each vector is RMS-normalized. This prevents the attention logits (`q · k / sqrt(head_dim)`) from growing large, which would cause the softmax to become extremely peaked (all attention on one token) and kill gradient flow. QK-norm was introduced in the ViT literature and has been shown to stabilize training of large transformers.

#### The `q_gain` parameter

```python
self.q_gain = nn.Parameter(torch.full((num_heads,), qk_gain_init, dtype=torch.float32))
...
q = q * self.q_gain.to(dtype=q.dtype)[None, :, None, None]
```

After RoPE and QK-norm, each query head is multiplied by a learnable scalar. Since QK-norm has set `||q|| = 1`, this `q_gain` effectively controls the **effective temperature** of the attention softmax for each head independently. Higher gain → more peaked attention (more selective). Lower gain → flatter attention (more averaged). This is initialized to 1.5 (slightly sharper than 1.0) and is trained by Adam as a scalar parameter. The model can learn that some heads should be broad and some should be sharp.

#### The output projection

```python
self.proj = CastedLinear(dim, dim, bias=False)
self.proj._zero_init = True
```

The output projection is initialized to **zero weights**. This means at the beginning of training, every attention layer outputs exactly zero — the residual connection passes the input unchanged through every layer. This is a deliberate initialization strategy: the model starts as an identity function and gradually learns to use each layer. This prevents the "lottery ticket at initialization" problem where some layers dominate early and others never activate.

---

### `MLP` (lines 606–617)

```python
class MLP(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        x = torch.relu(self.fc(x))
        return self.proj(x.square())
```

The activation function is **ReLU²** — ReLU followed by squaring: `output = max(0, x)²`. Compare to:

- **ReLU**: `max(0, x)` — simple, sparse, but gradient is flat for negative inputs
- **GELU/SiLU**: `x * sigmoid(x)` — smooth approximation to ReLU, commonly used in GPT-2/3/4
- **ReLU²**: `max(0, x)²` — sparser than GELU, amplifies strong activations more aggressively

The squaring operation means that activations near zero are suppressed more aggressively than in plain ReLU (a value of 0.1 after ReLU becomes 0.01 after squaring), while large activations are amplified (1.0 stays 1.0, 2.0 becomes 4.0). This creates **sparser** intermediate representations, which appears to help small models by reducing interference between features.

Like the attention output projection, `self.proj._zero_init = True` ensures the MLP starts as a no-op.

The expansion ratio is `mlp_mult = 2`, giving a hidden dimension of `512 × 2 = 1024`. This is narrower than the typical 4× expansion used in GPT-2, trading expressiveness for parameter count.

---

### `Block` (lines 620–645)

A `Block` is a single transformer layer. The standard structure is:

```
x = x + Attention(RMSNorm(x))
x = x + MLP(RMSNorm(x))
```

This script adds several enhancements:

#### Per-dimension output scaling

```python
self.attn_scale = nn.Parameter(torch.ones(dim, dtype=torch.float32))
self.mlp_scale  = nn.Parameter(torch.ones(dim, dtype=torch.float32))
```

Instead of adding the attention or MLP output directly, it is first element-wise multiplied by a learnable `dim`-dimensional scale vector:

```python
x = x + self.attn_scale.to(dtype=x.dtype)[None, None, :] * attn_out
x = x + self.mlp_scale.to(dtype=x.dtype)[None, None, :]  * self.mlp(self.mlp_norm(x))
```

These scales start at 1.0 (initialized as `torch.ones`) and are trained with Adam. The model can learn to suppress certain dimensions of the attention or MLP output — essentially learning to gate the contribution of each feature dimension independently.

#### The `resid_mix` parameter and the `x0` connection

```python
self.resid_mix = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))).float())
```

This is a `[2, dim]` parameter. At the start of each block's forward pass:

```python
mix = self.resid_mix.to(dtype=x.dtype)
x = mix[0][None, None, :] * x + mix[1][None, None, :] * x0
```

`mix[0]` starts at all-ones and `mix[1]` starts at all-zeros, so initially this is just `x = 1 * x + 0 * x0 = x`. But during training, the model can learn to blend in `x0` — the initial post-embedding representation — at each layer.

What is `x0`? It is the output of the embedding lookup and normalization at the very start of the forward pass, before any transformer layers. Passing `x0` through every block gives each layer a **direct shortcut to the input embedding**. This is similar in spirit to the LSTM's direct connection from input to output at each timestep. For small, deep networks, this kind of shortcut can help gradient flow and allow layers to "refresh" their representation with input information.

---

### `GPT` (lines 648–724)

The `GPT` class assembles all the modules and defines the full forward pass.

#### U-Net skip connections

```python
self.num_encoder_layers = num_layers // 2          # = 4 for num_layers=9
self.num_decoder_layers = num_layers - self.num_encoder_layers  # = 5
self.num_skip_weights   = min(self.num_encoder_layers, self.num_decoder_layers)  # = 4
self.skip_weights = nn.Parameter(torch.ones(self.num_skip_weights, model_dim, dtype=torch.float32))
```

The 9 layers are split into an "encoder" (first 4 layers) and a "decoder" (last 5 layers). During the forward pass:

```python
# Encoder: push outputs onto a stack
for i in range(self.num_encoder_layers):
    x = self.blocks[i](x, x0)
    skips.append(x)

# Decoder: pop from stack in reverse order
for i in range(self.num_decoder_layers):
    if skips:
        x = x + self.skip_weights[i].to(dtype=x.dtype)[None, None, :] * skips.pop()
    x = self.blocks[self.num_encoder_layers + i](x, x0)
```

Since `skips` is a stack (LIFO — Last In, First Out), `skips.pop()` in the decoder gives the skip connections in **reverse** encoder order. So:
- Decoder layer 0 (= block 4) receives a skip from Encoder layer 3 (= block 3)
- Decoder layer 1 (= block 5) receives a skip from Encoder layer 2 (= block 2)
- Decoder layer 2 (= block 6) receives a skip from Encoder layer 1 (= block 1)
- Decoder layer 3 (= block 7) receives a skip from Encoder layer 0 (= block 0)
- Decoder layer 4 (= block 8) receives no skip (stack is empty)

This creates U-Net-style long-range connections: the deepest encoder layer connects to the shallowest decoder layer, like the symmetric "U" shape in image segmentation networks. The intuition is that early layers capture basic syntactic patterns and late layers capture high-level semantics; the skip connection allows the decoder to combine high-level context with lower-level features.

`skip_weights` is a learnable `[4, dim]` tensor, initialized to ones, trained by Adam. Each decoder layer can independently scale the contribution of its corresponding encoder skip, and can scale it per-dimension.

#### Tied vs. untied embeddings

```python
self.tok_emb = nn.Embedding(vocab_size, model_dim)
self.lm_head = None if tie_embeddings else CastedLinear(model_dim, vocab_size, bias=False)
```

When `tie_embeddings=True`:
- `lm_head` is `None`
- In the forward pass: `logits_proj = F.linear(x, self.tok_emb.weight)`

The same `[vocab_size, model_dim]` matrix is used for both the input embedding (mapping token IDs to vectors) and the output projection (mapping hidden states back to vocab logit scores). The `F.linear` call uses it transposed in the output direction — it is `x @ tok_emb.weight.T`.

**Cost savings:** With `vocab_size=1024` and `model_dim=512`, tying saves `1024 × 512 = 524,288` parameters — about 2 MB in fp32. That is a meaningful fraction of a 16 MB budget.

**Trade-off:** A single matrix must serve two very different purposes simultaneously. The embedding rows represent "what does this token mean?" while the output weight rows represent "how likely is this token as the next word?". These are related but not identical. In practice, tying works well for small models where parameter efficiency matters more than expressiveness.

#### Weight initialization

```python
def _init_weights(self) -> None:
    if self.tie_embeddings:
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=self.tied_embed_init_std)
    for module in self.modules():
        if isinstance(module, nn.Linear) and getattr(module, "_zero_init", False):
            nn.init.zeros_(module.weight)
```

Two initialization strategies:

1. **Tiny normal for tied embeddings:** `std=0.005` is very small (default PyTorch embedding init is `std=1.0`). This is because the tied embedding is used both as input and output. If initialized at the standard scale of 1.0, the initial logits would be enormous (`x @ tok_emb.weight.T` with unit-scale vectors produces unit-scale logits, and softmax of those is very noisy). The small initialization keeps the model well-behaved from the first step.

2. **Zero initialization for output projections:** Every `CastedLinear` marked with `_zero_init = True` (the output projections of attention and MLP, plus `lm_head` when untied) is initialized to exactly zero. Combined with the identity-like `resid_mix` initialization, this means the model starts as an identity function: input → embedding → no-op blocks → output.

#### The forward pass

```python
def forward(self, input_ids: Tensor, target_ids: Tensor) -> Tensor:
    x  = self.tok_emb(input_ids)    # embed: [batch, seq, dim]
    x  = F.rms_norm(x, ...)         # normalize embedding
    x0 = x                          # save initial embedding for resid_mix
    # ... encoder blocks ...
    # ... decoder blocks with skips ...
    x = self.final_norm(x).reshape(-1, x.size(-1))
    targets = target_ids.reshape(-1)
    logits_proj = F.linear(x, self.tok_emb.weight)  # or lm_head(x) if untied
    logits = self.logit_softcap * torch.tanh(logits_proj / self.logit_softcap)
    return F.cross_entropy(logits.float(), targets, reduction="mean")
```

**Logit softcap:** After computing logits, they are passed through:
```
logits = cap * tanh(logits / cap)
```
with `cap = 30.0`. This bounds the logits to `[-30, 30]` regardless of weight magnitudes. Without this, logits can grow very large late in training, making the softmax near-deterministic and the model overconfident. The softcap was used in Google's Gemma 2 model and is especially helpful for tied embeddings where the logit scale can become misaligned with the embedding scale.

**Returns loss, not logits:** The model's `forward()` returns the scalar cross-entropy loss directly. This is convenient for the training loop (you can call `loss.backward()` immediately) and prevents accidentally forgetting to apply the loss function.

<a id="section-8"></a>

---

## Section 8: main() — Distributed & CUDA Setup (lines 731–798)

Everything from here to the end of the file lives inside the `main()` function, which is called at the bottom via:

```python
if __name__ == "__main__":
    main()
```

The script is launched with `torchrun`, PyTorch's tool for spawning one process per GPU:

```bash
torchrun --nproc_per_node=8 train_gpt.py
```

`torchrun` spawns 8 separate Python processes (one per GPU), sets the environment variables `RANK`, `WORLD_SIZE`, and `LOCAL_RANK` for each, and they all run the same `main()` function. The differences in behavior between ranks come from reading those environment variables.

---

### Compiling the Newton-Schulz function

```python
global zeropower_via_newtonschulz5
zeropower_via_newtonschulz5 = torch.compile(zeropower_via_newtonschulz5)
```

The first thing `main()` does is compile the Newton-Schulz function with `torch.compile`. This turns the Python function into an optimized CUDA kernel the first time it is called. Because it is called many times per training step (once per matrix parameter per Muon step), the JIT-compiled version is significantly faster than the eager Python version.

The `global` declaration allows reassigning the module-level name. After this line, all calls to `zeropower_via_newtonschulz5` within `Muon.step()` will use the compiled version.

---

### Reading distributed environment variables

```python
distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
rank       = int(os.environ.get("RANK",       "0"))
world_size = int(os.environ.get("WORLD_SIZE", "1"))
local_rank = int(os.environ.get("LOCAL_RANK", "0"))
```

- `RANK` — this process's global rank (0 to world_size-1)
- `WORLD_SIZE` — total number of processes (number of GPUs)
- `LOCAL_RANK` — this process's rank within its node (0 to nproc_per_node-1)

For a single-node 8-GPU setup, `RANK == LOCAL_RANK`. For multi-node setups, `RANK` is global and `LOCAL_RANK` identifies which GPU on this machine to use.

---

### Gradient accumulation setup

```python
if 8 % world_size != 0:
    raise ValueError("WORLD_SIZE must divide 8 so grad_accum_steps stays integral")
grad_accum_steps = 8 // world_size
```

The global batch size is always 524,288 tokens, spread across `8` "virtual GPUs" (whether they are physical GPUs or accumulation steps). With 8 real GPUs, each handles one virtual GPU's share in a single forward pass. With 4 GPUs, each does 2 accumulation steps. With 1 GPU, it does 8 accumulation steps.

This design constraint — that `world_size` must divide 8 — keeps the math exact. The comment explains why: `grad_accum_steps` must be a whole number.

---

### CUDA and distributed initialization

```python
device = torch.device("cuda", local_rank)
torch.cuda.set_device(device)
if distributed:
    dist.init_process_group(backend="nccl", device_id=device)
    dist.barrier()
master_process = rank == 0
```

Each process sets its active CUDA device to its local GPU. `dist.init_process_group` initializes NCCL, the GPU-aware collective communication library used for `all_reduce` and `barrier`. `dist.barrier()` ensures all processes have finished initialization before any of them proceed — a synchronization point.

`master_process = (rank == 0)` is the pattern for "only do this on one GPU." Log writes, file saves, and progress messages all use `master_process` to avoid redundant output from all 8 processes.

---

### Fast math settings

```python
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
```

TF32 (TensorFloat-32) is an NVIDIA format that stores 10 bits of mantissa (vs. 23 for fp32 and 7 for fp16). On Ampere and later GPUs, matrix multiplications can use TF32 at full tensor-core throughput while retaining more precision than fp16. These two lines enable it for both cuDNN operations and direct CUDA matmuls. The quality impact is negligible for training.

```python
enable_cudnn_sdp(False)
enable_flash_sdp(True)
enable_mem_efficient_sdp(False)
enable_math_sdp(False)
```

PyTorch's `scaled_dot_product_attention` has four possible backends. This configuration forces **Flash Attention** exclusively. Flash Attention uses a tiled algorithm that avoids materializing the full `[batch, heads, seq, seq]` attention matrix in global GPU memory, which is crucial for long sequences. It also fuses the attention computation into a single kernel, reducing memory bandwidth usage. On H100s with large sequences, Flash Attention is the fastest option by a significant margin. Disabling the other backends prevents PyTorch from falling back to a slower implementation.

---

### Logging setup

```python
def log0(msg: str, console: bool = True) -> None:
    if not master_process:
        return
    if console:
        print(msg)
    if logfile is not None:
        with open(logfile, "a", encoding="utf-8") as f:
            print(msg, file=f)
```

`log0` is a small helper that only outputs from rank 0. The `console=False` option allows writing to the log file without cluttering the terminal (used for the full source code dump and system info). The logfile name is `logs/{run_id}.txt`, where `run_id` is the UUID generated at startup.

The very first thing written to the log is the complete source code of `train_gpt.py` itself:

```python
log0(code, console=False)
```

This means every log file is fully self-documenting — you can always recover the exact code that produced any logged result.

<a id="section-9"></a>

---

## Section 9: Tokenizer & Validation Setup (lines 800–820)

This section seeds all random number generators, loads the tokenizer, loads the validation data, and builds the BPB lookup tables.

---

### Seeding all RNGs

```python
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
```

Four separate RNG states are seeded:

- `random.seed` — Python's built-in RNG (not used much here, but good hygiene)
- `np.random.seed` — NumPy's global RNG
- `torch.manual_seed` — PyTorch CPU RNG (used for weight initialization)
- `torch.cuda.manual_seed_all` — PyTorch CUDA RNG for **all GPUs** on this machine

Seeding all four ensures that two runs with the same `seed` and the same hardware configuration will produce bit-identical results. This is critical for debugging: if you observe a bad training run, you can reproduce it exactly and step through the first few iterations.

Note that seeding is done **before** model creation. This is important because `_init_weights()` calls `nn.init.normal_()` and `nn.init.zeros_()`, which consume from the PyTorch RNG. If you seed after creating the model, the initialization would differ between runs.

---

### Tokenizer loading and validation

```python
sp = spm.SentencePieceProcessor(model_file=args.tokenizer_path)
if int(sp.vocab_size()) != args.vocab_size:
    raise ValueError(...)
```

The SentencePiece tokenizer is loaded from the `.model` file. The vocab size is validated against the hyperparameter — a mismatch means the model would have an embedding table of the wrong size relative to what the tokenizer produces.

This is a defensive check that catches a common mistake: using a tokenizer trained for a different vocabulary size. The error is caught here, before any model weights are created, with a clear message.

---

### Validation data and BPB tables

```python
val_tokens = load_validation_tokens(args.val_files, args.train_seq_len)
base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = build_sentencepiece_luts(
    sp, args.vocab_size, device
)
```

`load_validation_tokens` reads all validation shards into a single large tensor (stored on CPU). The three lookup tables are built on the GPU (passed `device`) because they will be used in the inner validation loop where every microsecond counts.

The logging that follows confirms what was loaded:

```python
log0(f"val_bpb:enabled tokenizer_kind=sentencepiece tokenizer_path={args.tokenizer_path}")
log0(f"train_loader:dataset:{dataset_dir.name} train_shards:{actual_train_files}")
log0(f"val_loader:shards pattern={args.val_files} tokens:{val_tokens.numel() - 1}")
```

The `-1` in the token count is because `val_tokens` has one extra token (the target shift offset), so the actual number of evaluable tokens is `numel() - 1`.

<a id="section-10"></a>

---

## Section 10: Model & Optimizer Setup (lines 822–910)

This section creates the model, moves it to the GPU with the right precision strategy, compiles it, wraps it in DDP, and constructs the optimizers.

---

### Model creation and precision setup

```python
base_model = GPT(...).to(device).bfloat16()
for module in base_model.modules():
    if isinstance(module, CastedLinear):
        module.float()
restore_low_dim_params_to_fp32(base_model)
```

The model is created on CPU, then moved to GPU (`.to(device)`) and cast to `bfloat16` (`.bfloat16()`). After this call, every parameter in the model is on the correct device and in bf16.

Then two targeted precision restores happen:

1. **`CastedLinear.float()`** — Every `CastedLinear` module (all the weight matrices: Q, K, V, output projections, MLP layers) has its weight restored to fp32. Remember that `CastedLinear.forward()` casts to the input dtype at matmul time, so these fp32 weights will still be used in bf16 computation — but they are stored in fp32 so the optimizer can apply precise updates.

2. **`restore_low_dim_params_to_fp32(base_model)`** — All 1D parameters and control tensors are restored to fp32, as discussed in Section 7.

The final precision state is:
- `CastedLinear` weights (large matrices) — **fp32** (for optimizer precision, bf16 at compute time)
- Control tensors (`attn_scale`, `mlp_scale`, `resid_mix`, `q_gain`, `skip_weights`) — **fp32**
- All other parameters (embedding, etc.) — **bf16**

---

### Compilation and DDP wrapping

```python
compiled_model = torch.compile(base_model, dynamic=False, fullgraph=True)
model = DDP(compiled_model, device_ids=[local_rank], broadcast_buffers=False) if distributed else compiled_model
```

`torch.compile(dynamic=False, fullgraph=True)` compiles the entire model graph into a single optimized CUDA program. Key flags:

- **`dynamic=False`** — assumes input shapes are constant. This allows more aggressive kernel fusion but will recompile if shapes change (they don't change here since `seq_len` is fixed).
- **`fullgraph=True`** — requires the entire forward pass to be captured as a single graph. This fails loudly if any Python control flow based on tensor values (data-dependent branching) is present. It is a correctness check: if your model has data-dependent branches, compilation will error rather than silently produce wrong results.

`DDP` wraps the compiled model for gradient synchronization. `broadcast_buffers=False` is important: DDP by default broadcasts non-parameter buffers (like the RoPE cache in `Rotary`) at the start of each step, but these are deterministically computed and identical on all ranks, so broadcasting them wastes bandwidth.

Note that `base_model` is kept as a direct reference to the unwrapped GPT object. This is used for optimizer setup (which needs raw parameters) and for saving the state dict. `model` (the DDP-wrapped version) is used for the actual forward/backward passes.

---

### Optimizer parameter groups

This is a critical and interesting part of the setup. The parameters are split into groups with different optimizers and learning rates:

#### Matrix parameters → Muon

```python
block_named_params = list(base_model.blocks.named_parameters())
matrix_params = [
    p for name, p in block_named_params
    if p.ndim == 2 and not any(pattern in name for pattern in CONTROL_TENSOR_NAME_PATTERNS)
]
```

All 2D tensors in the transformer blocks that are NOT control tensors. These are the Q, K, V, output projection, and MLP weight matrices. They go to the Muon optimizer.

#### Scalar parameters → Adam

```python
scalar_params = [
    p for name, p in block_named_params
    if p.ndim < 2 or any(pattern in name for pattern in CONTROL_TENSOR_NAME_PATTERNS)
]
if base_model.skip_weights.numel() > 0:
    scalar_params.append(base_model.skip_weights)
```

All 1D parameters (vectors) and control tensors from the blocks, plus the `skip_weights`. These go to the Adam optimizer. Even though `resid_mix` and `skip_weights` are technically 2D, they are control tensors and explicitly go to Adam — Muon's orthogonalization would be incorrect for these parameters since they have a different semantic role than weight matrices.

#### Embedding → Adam with its own LR

```python
token_lr = args.tied_embed_lr if args.tie_embeddings else args.embed_lr
optimizer_tok = torch.optim.Adam(
    [{"params": [base_model.tok_emb.weight], "lr": token_lr, "base_lr": token_lr}],
    betas=(args.beta1, args.beta2), eps=args.adam_eps, fused=True,
)
```

The embedding matrix gets its own optimizer with a different learning rate. When tied, `tied_embed_lr=0.05` is used (lower than the untied `embed_lr=0.6`) because the tied embedding must also serve as the output head — too aggressive an update would destabilize the output logit scale.

The `"base_lr"` key stored in each param group is not a standard PyTorch optimizer field — it is a custom field used to implement the LR warmdown schedule. The actual `"lr"` is set each step to `base_lr * scale`, where `scale` comes from `lr_mul()`. Storing `base_lr` separately allows the schedule to always scale from the original hyperparameter value rather than accumulating drift.

`fused=True` uses a CUDA-fused Adam implementation that performs all update operations (momentum, variance, parameter update) in a single kernel launch, reducing overhead for many small parameter groups.

#### Untied head → Adam with its own LR

```python
if base_model.lm_head is not None:
    optimizer_head = torch.optim.Adam(
        [{"params": [base_model.lm_head.weight], "lr": args.head_lr, "base_lr": args.head_lr}],
        ...
    )
    optimizers.insert(1, optimizer_head)
```

If `tie_embeddings=False`, a separate output head exists and gets its own optimizer with `head_lr=0.008`. This is lower than the embedding LR because the output head is closer to the loss and can benefit from more conservative updates.

---

### The `optimizers` list

```python
optimizers: list[torch.optim.Optimizer] = [optimizer_tok, optimizer_muon, optimizer_scalar]
```

All optimizers are collected into a list. The training loop always calls `opt.step()` and `opt.zero_grad()` for every optimizer in this list. This unified handling means adding or removing an optimizer group requires only changing this list.

<a id="section-11"></a>

---

## Section 11: Compilation Warmup (lines 912–961)

`torch.compile` compiles lazily — the CUDA kernels are generated the first time a given input shape is seen. For a large model, this compilation can take **several minutes**. If the competition measures a hard 10-minute wallclock cap, that compilation time would count against training, severely penalizing the first run while subsequent runs (which reuse the compiled cache) would be much faster.

The solution is a **throwaway warmup phase**: run a fixed number of steps to trigger compilation, then **restore the original model weights and optimizer state** as if training had never happened. Real training then starts from a clean initialization, and the 10-minute clock starts after compilation is complete.

---

### Saving initial state

```python
if args.warmup_steps > 0:
    initial_model_state    = {name: tensor.detach().cpu().clone()
                              for name, tensor in base_model.state_dict().items()}
    initial_optimizer_states = [copy.deepcopy(opt.state_dict()) for opt in optimizers]
```

Before any warmup step, the model's entire state dict is cloned to CPU memory. Optimizer states are deep-copied. These snapshots will be restored after warmup.

The CPU clone is important: if the snapshot were kept on GPU, it would double the GPU memory usage during warmup. Since GPU memory is at a premium for large models, pushing the snapshot to CPU is the right trade-off.

---

### The warmup loop

```python
model.train()
for warmup_step in range(args.warmup_steps):
    zero_grad_all()
    for micro_step in range(grad_accum_steps):
        if distributed:
            model.require_backward_grad_sync = micro_step == grad_accum_steps - 1
        x, y = train_loader.next_batch(args.train_batch_tokens, args.train_seq_len, grad_accum_steps)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
            warmup_loss = model(x, y)
        (warmup_loss * grad_scale).backward()
    for opt in optimizers:
        opt.step()
    zero_grad_all()
```

This is structurally identical to the real training loop. It runs `warmup_steps=20` full steps (including gradient accumulation, backward pass, and optimizer updates) to ensure all code paths are compiled. The compiled kernels cache on disk (in PyTorch's compilation cache directory), so they are reused in future runs on the same hardware.

The `model.require_backward_grad_sync` flag controls DDP's gradient synchronization. Setting it to `False` on non-final micro-steps tells DDP to skip the `all_reduce` of gradients until the last accumulation step. This matches the real training behavior and ensures the same communication pattern is compiled.

---

### Restoring initial state

```python
base_model.load_state_dict(initial_model_state, strict=True)
for opt, state in zip(optimizers, initial_optimizer_states, strict=True):
    opt.load_state_dict(state)
zero_grad_all()
if distributed:
    model.require_backward_grad_sync = True
train_loader = DistributedTokenLoader(args.train_files, rank, world_size, device)
```

After warmup, the model weights are restored to their initial values, the optimizer momentum and variance buffers are cleared to their initial state (all zeros for Adam, no buffers for Muon), and the data loader is recreated so training data starts from the beginning.

This is a complete reset. The compiled kernels remain cached in PyTorch's internals, but from the model's perspective it is as if warmup never happened.

The `strict=True` argument to `load_state_dict` ensures the restored state dict has exactly the same keys as the current model — any mismatch causes an error rather than silently loading partial weights.

---

### Why 20 warmup steps?

With `dynamic=False, fullgraph=True`, `torch.compile` only needs to see **one** input shape to compile all kernels (since shapes never change). A single step would suffice for compilation. The 20 steps provide a small buffer — some optimizers or ops may have deferred initialization that only triggers after the first step, so a few extra steps ensure everything has been visited. The default of 20 is conservative but adds only a trivial amount of time compared to the multi-minute compilation itself.

<a id="section-12"></a>

---

## Section 12: Main Training Loop (lines 963–1060)

The training loop is the heart of the script. It runs continuously until either the step limit or the wallclock cap is reached, performing validation at specified intervals.

---

### Loop setup

```python
training_time_ms = 0.0
stop_after_step: int | None = None
torch.cuda.synchronize()
t0 = time.perf_counter()

step = 0
while True:
```

`training_time_ms` accumulates total training time, excluding validation passes (validation is paused out of the timer). The `torch.cuda.synchronize()` before setting `t0` ensures the GPU has finished any pending operations from warmup before we start timing.

`stop_after_step` starts as `None`. When the wallclock cap is hit, it is set to the current step number. The loop then runs one more iteration to do a final validation, then breaks.

---

### Loop termination condition

```python
last_step = step == args.iterations or (stop_after_step is not None and step >= stop_after_step)
```

`last_step` is `True` when either:
- The step counter has reached `iterations = 20000`, or
- The wallclock cap has been reached and we have just completed the final step

The loop structure is: check `last_step`, validate if needed, break if `last_step`, otherwise train.

---

### Validation

```python
should_validate = last_step or (args.val_loss_every > 0 and step % args.val_loss_every == 0)
if should_validate:
    torch.cuda.synchronize()
    training_time_ms += 1000.0 * (time.perf_counter() - t0)
    val_loss, val_bpb = eval_val(...)
    log0(f"step:{step}/{args.iterations} val_loss:{val_loss:.4f} val_bpb:{val_bpb:.4f} ...")
    torch.cuda.synchronize()
    t0 = time.perf_counter()
```

Validation runs at step 0 (before any training), every `val_loss_every=1000` steps, and on the final step. The timer is **paused** around the validation call — `training_time_ms` accumulates only actual training time. After validation, the timer is **restarted** from the current moment.

This timer pausing is essential for the wallclock-aware LR schedule (discussed below) to be accurate: if validation time were counted as training time, the schedule would think more time had elapsed than actually had.

---

### The learning rate schedule: `lr_mul()`

```python
def lr_mul(step: int, elapsed_ms: float) -> float:
    if args.warmdown_iters <= 0:
        return 1.0
    if max_wallclock_ms is None:
        warmdown_start = max(args.iterations - args.warmdown_iters, 0)
        return max((args.iterations - step) / max(args.warmdown_iters, 1), 0.0) \
               if warmdown_start <= step < args.iterations else 1.0
    step_ms = elapsed_ms / max(step, 1)
    warmdown_ms = args.warmdown_iters * step_ms
    remaining_ms = max(max_wallclock_ms - elapsed_ms, 0.0)
    return remaining_ms / max(warmdown_ms, 1e-9) if remaining_ms <= warmdown_ms else 1.0
```

This function returns a multiplier `[0, 1]` for the current learning rate.

**Without wallclock cap** (for step-based training): LR is 1.0 (full) for the first `iterations - warmdown_iters` steps, then linearly decays to 0 over the final `warmdown_iters` steps.

**With wallclock cap** (the default): The schedule adapts to elapsed *time*, not step count. It estimates how long each step takes (`step_ms = elapsed_ms / step`) and calculates how many milliseconds the warmdown period would take (`warmdown_ms = warmdown_iters * step_ms`). If the remaining time is less than `warmdown_ms`, it starts warming down proportionally so the LR reaches 0 exactly when the 10 minutes expire.

This is a sophisticated design: the LR schedule is time-aware. If the GPU is slower or faster than expected, the warmdown automatically adjusts. The result is always an LR that gracefully descends to 0 at the exact end of the run, which is known to improve final model quality compared to stopping training with a non-zero LR.

---

### The training step

```python
elapsed_ms = training_time_ms + 1000.0 * (time.perf_counter() - t0)
scale = lr_mul(step, elapsed_ms)
zero_grad_all()
train_loss = torch.zeros((), device=device)

for micro_step in range(grad_accum_steps):
    if distributed:
        model.require_backward_grad_sync = micro_step == grad_accum_steps - 1
    x, y = train_loader.next_batch(args.train_batch_tokens, args.train_seq_len, grad_accum_steps)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
        loss = model(x, y)
    train_loss += loss.detach()
    (loss * grad_scale).backward()

train_loss /= grad_accum_steps
```

**Gradient accumulation:** The inner loop runs `grad_accum_steps` times. Each micro-step:
1. Loads a batch
2. Runs the forward pass in bf16 autocast
3. Scales the loss by `grad_scale = 1.0 / grad_accum_steps` before calling `.backward()`

The loss scaling ensures that the accumulated gradients are equivalent to computing gradients on a single batch of `global_batch_size` tokens. Without the scaling, the gradients would be `grad_accum_steps` times too large.

**DDP gradient sync suppression:** `model.require_backward_grad_sync = False` on all but the last micro-step tells DDP to skip the `all_reduce` of gradients during intermediate micro-steps. This is a critical optimization: without it, each of the 8 accumulation steps would trigger an `all_reduce`, adding 7 redundant communication rounds. With it, only the final micro-step triggers the `all_reduce`.

---

### Muon momentum warmup

```python
frac = min(step / args.muon_momentum_warmup_steps, 1.0) if args.muon_momentum_warmup_steps > 0 else 1.0
muon_momentum = (1 - frac) * args.muon_momentum_warmup_start + frac * args.muon_momentum
for group in optimizer_muon.param_groups:
    group["momentum"] = muon_momentum
```

Muon's momentum starts at `muon_momentum_warmup_start=0.85` and linearly increases to `muon_momentum=0.95` over the first `muon_momentum_warmup_steps=500` steps. Lower momentum early in training means the optimizer is more responsive to the current gradient and less dominated by accumulated history. This is helpful during the initial phase when the model's gradients are large and rapidly changing.

---

### Applying the LR schedule

```python
for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["base_lr"] * scale
```

This iterates over all optimizers and all their parameter groups. Each group's `"lr"` is set to `base_lr * scale`. Since `scale` is computed before the optimizer step, this correctly applies the warmdown multiplier to all four optimizers simultaneously.

---

### Wallclock cap enforcement

```python
approx_training_time_ms = training_time_ms + 1000.0 * (time.perf_counter() - t0)
reached_cap = max_wallclock_ms is not None and approx_training_time_ms >= max_wallclock_ms
if distributed and max_wallclock_ms is not None:
    reached_cap_tensor = torch.tensor(int(reached_cap), device=device)
    dist.all_reduce(reached_cap_tensor, op=dist.ReduceOp.MAX)
    reached_cap = bool(reached_cap_tensor.item())
if stop_after_step is None and reached_cap:
    stop_after_step = step
```

After each training step, the elapsed time is checked against the 10-minute cap. In distributed mode, the `reached_cap` boolean is communicated across all ranks via `all_reduce(MAX)`. This ensures that if **any** rank thinks the cap has been reached, all ranks agree to stop together. The `MAX` operation means that `True` (=1) from any rank propagates to all ranks.

When `stop_after_step` is set, the loop does not immediately stop. It runs one more iteration: the `last_step` check at the top of the loop triggers a final validation pass (with the quantized model after training), then breaks.

<a id="section-13"></a>

---

## Section 13: Serialization & Roundtrip Validation (lines 1062–1126)

After training completes, the model is serialized to disk in two formats and a final validation is run on the compressed model to establish the official submission score.

---

### Raw save (lines 1068–1074)

```python
if master_process:
    torch.save(base_model.state_dict(), "final_model.pt")
    model_bytes = os.path.getsize("final_model.pt")
    code_bytes   = len(code.encode("utf-8"))
    log0(f"Serialized model: {model_bytes} bytes")
    log0(f"Code size: {code_bytes} bytes")
    log0(f"Total submission size: {model_bytes + code_bytes} bytes")
```

The raw state dict is saved to `final_model.pt`. This file contains every parameter in its training precision (fp32 for matrices, fp16 for embeddings, etc.) and will be much larger than 16 MB — it is only useful for debugging, reloading, or fine-tuning.

The code size (`code_bytes`) is also logged. The competition submission is the model file **plus** the Python script, and the total must be under 16 MB. The script itself at ~1,126 lines is around 40–50 KB, which is negligible.

---

### Int8 + zlib export (lines 1076–1092)

```python
quant_obj, quant_stats = quantize_state_dict_int8(base_model.state_dict())
quant_buf = io.BytesIO()
torch.save(quant_obj, quant_buf)
quant_raw = quant_buf.getvalue()
quant_blob = zlib.compress(quant_raw, level=9)
```

The quantization pipeline from Section 5 is applied to the entire state dict. The result (`quant_obj`) is a nested Python dict containing separate sub-dicts for int8 tensors, scales, and passthrough tensors. This is serialized with `torch.save` into an in-memory buffer (`io.BytesIO`), then the raw bytes are compressed with `zlib.compress(level=9)`.

Using `io.BytesIO` as the target of `torch.save` avoids writing an uncompressed intermediate file — the uncompressed blob lives only in memory, and the compressed blob is written directly to disk.

```python
if master_process:
    with open("final_model.int8.ptz", "wb") as f:
        f.write(quant_blob)
```

The `.ptz` extension is a convention for compressed PyTorch tensors (`.pt` + zlib = `.ptz`). This file is the actual competition submission artifact.

The stats logged include:
- `payload_bytes` — total size of all int8 tensors and scales (before torch.save overhead)
- `raw_torch` — size of the serialized-but-uncompressed blob
- `payload_ratio` — compression ratio compared to the original fp32/bf16 model

---

### Roundtrip validation (lines 1094–1119)

This is the most important part of the serialization section. The official score is not the training-time BPB — it is the BPB **of the quantized model after roundtrip through disk**.

```python
if distributed:
    dist.barrier()
with open("final_model.int8.ptz", "rb") as f:
    quant_blob_disk = f.read()
quant_state = torch.load(io.BytesIO(zlib.decompress(quant_blob_disk)), map_location="cpu")
base_model.load_state_dict(dequantize_state_dict_int8(quant_state), strict=True)
```

The file is read back from disk, decompressed, deserialized, dequantized, and loaded back into the model. Every rank reads the same file. The `dist.barrier()` before reading ensures all ranks wait for rank 0 to finish writing the file before any rank tries to read it.

```python
q_val_loss, q_val_bpb = eval_val(...)
log0(f"final_int8_zlib_roundtrip val_loss:{q_val_loss:.4f} val_bpb:{q_val_bpb:.4f} ...")
log0(f"final_int8_zlib_roundtrip_exact val_loss:{q_val_loss:.8f} val_bpb:{q_val_bpb:.8f}")
```

The full validation is re-run on the dequantized model. The result is logged at 4 decimal places for readability and **8 decimal places** for the official score. The line containing `final_int8_zlib_roundtrip_exact` is the authoritative competition score.

**Why validate the roundtripped model?**

Quantization is lossy. The dequantized weights are `int8_value * scale`, which is not the same as the original float weights. The quantization error — how much the model's BPB degrades due to quantization — is part of the submission's quality. A model that compresses well but degrades severely under quantization would have a worse competition score than shown by the pre-quantization validation. The roundtrip validation makes this explicit and honest.

The difference between the final training-time `val_bpb` and the roundtrip `q_val_bpb` indicates how well the model survived quantization. A small difference (< 0.005 BPB) means the quantization was high quality. A larger difference suggests the model has high quantization sensitivity (often due to outlier weights).

---

### Cleanup (lines 1121–1122)

```python
if distributed:
    dist.destroy_process_group()
```

The distributed process group is destroyed at the end. This is good hygiene — it allows `torchrun` to cleanly shut down the NCCL communicators and release GPU resources. Without this, the processes might hang waiting for peers that have already exited.

<a id="summary"></a>

---

## Summary: The Full Data Flow

Here is the complete end-to-end flow from raw text to competition score:

### 1. Data preparation (offline, before training)

FineWeb web-crawl text is tokenized with a SentencePiece BPE tokenizer trained to a vocabulary of 1024 tokens. The tokenized data is written to binary shard files in uint16 format with a standard 256-int32 header. Validation documents are fixed and always written to separate `fineweb_val_*.bin` shards.

### 2. Startup and setup

`torchrun` spawns 8 processes, one per H100 GPU. Each process:
- Reads all hyperparameters from environment variables
- Initializes NCCL for inter-GPU communication
- Sets up Flash Attention as the only SDP backend
- Seeds all RNGs identically

### 3. Model construction

A GPT-style transformer is built with:
- Token embedding: `1024 × 512` (shared as output head when tied)
- 9 blocks, each with GQA (8Q/4KV heads), RoPE, QK-norm, q_gain, ReLU² MLP, `attn_scale`, `mlp_scale`, and `resid_mix`
- U-Net skip connections between encoder layers (0–3) and decoder layers (4–7)
- Logit softcap at ±30

Weights are stored in fp32 for matrices and control tensors (optimizer precision), with bf16 used for compute.

### 4. Compilation warmup

`torch.compile` compiles the model and the Newton-Schulz function. 20 throwaway steps trigger compilation. Model weights and optimizer state are then restored to their initial values.

### 5. Training

The main loop runs for up to 20,000 steps or 10 minutes, whichever comes first.

Each step:
1. Compute the LR schedule multiplier (time-aware warmdown)
2. Run gradient accumulation (1 step on 8 GPUs, 8 micro-steps on 1 GPU)
3. Apply Muon to matrix parameters (with momentum warmup)
4. Apply Adam to embeddings, scalars, and control tensors
5. Log training loss every 200 steps, validate every 1000 steps

### 6. Export

After training:
1. Save raw state dict as `final_model.pt` (debugging only)
2. Quantize to int8 (per-row for matrices, per-tensor for vectors)
3. Serialize with `torch.save`, compress with `zlib(level=9)`
4. Write to `final_model.int8.ptz`

### 7. Scoring

The quantized model is loaded back from disk, dequantized, and evaluated on the full validation set. The resulting `val_bpb` logged on the `final_int8_zlib_roundtrip_exact` line is the official competition score.

---

## Levers for Improvement

The script is a solid baseline, but it leaves many directions open for exploration. Here is where participants have the most room to improve:

### Architecture

- **Depth vs. width trade-off** — the default is 9 layers × 512 dim. Narrower and deeper (12 layers × 384 dim) vs. shallower and wider (6 layers × 640 dim) for the same parameter count may behave differently under quantization and training dynamics.
- **Attention variants** — multi-query attention (1 KV head), sliding window attention, linear attention, or Mamba-style recurrence all occupy the same parameter budget differently.
- **Activation functions** — SwiGLU (used in Llama) expands to an intermediate dim with a gating mechanism. It is typically better than ReLU² but costs ~33% more parameters in the MLP for the same output dimension.
- **Embedding dimension per layer** — mixture-of-experts, width-varying layers, or parameter-shared layers could distribute the parameter budget non-uniformly across depth.

### Optimizer

- **Learning rates** — the four separate learning rates (embedding, head, matrix, scalar) are strongly interdependent and were tuned for the default architecture. Any architecture change likely requires re-tuning all four.
- **Muon tuning** — the number of Newton-Schulz steps (`backend_steps`), the convergence coefficients, and the momentum schedule all affect optimization quality.
- **LR schedule shape** — the script uses a flat-then-linear-warmdown schedule. Cosine warmdown, or a warmup period at the start, may improve early or late training dynamics.

### Quantization

- **Different bit widths** — 4-bit quantization with careful calibration can compress much more aggressively at some quality cost.
- **Outlier suppression** — techniques like SmoothQuant can redistribute weight magnitudes to reduce per-row variance before quantization, improving int8 accuracy.
- **Better compression formats** — the zlib step could be replaced with LZMA or brotli for higher compression ratios, squeezing more bytes out of the artifact.

### Tokenizer

- **Different vocabulary sizes** — a vocabulary of 512 gives more parameter budget to transformer layers but produces longer sequences. A vocabulary of 2048 or 4096 allows richer tokens but consumes more of the embedding budget. There is an optimal point depending on model capacity and training time.
- **Different tokenization algorithms** — character-level (vocab 256), BPE, unigram, or byte-pair-encoding with different merge counts all produce different token-to-byte ratios.

### Training dynamics

- **Sequence length** — longer sequences see more context but are more expensive. `seq_len=2048` may improve quality for tasks requiring long-range dependencies.
- **Batch size** — the 524,288-token batch is large. Smaller batches with more steps and lower LR, or larger batches with higher LR, explore different noise-generalization trade-offs.
- **Curriculum learning** — starting with shorter sequences or simpler data and transitioning to full-length sequences or harder data can accelerate early training.
- **Data mixture** — the FineWeb dataset is high quality but not curated by domain. Mixing in code, math, or other specialized text may improve BPB on the general validation set if the tokenizer was trained to represent those domains efficiently.

---

## Closing Notes

`train_gpt.py` demonstrates a significant number of modern techniques packed into a single readable file:

- **Muon** for matrix optimization (Section 3)
- **BPB** for tokenizer-agnostic evaluation (Section 4)
- **Int8 + zlib** for model compression (Section 5)
- **GQA + RoPE + QK-norm + ReLU²** in the transformer (Section 7)
- **U-Net skip connections + resid_mix + q_gain** for expressiveness (Section 7)
- **Tied embeddings + logit softcap** for parameter efficiency (Section 7)
- **Torch.compile + DDP** for training throughput (Sections 8, 10)
- **Time-aware LR warmdown** for optimal final quality (Section 12)
- **Roundtrip validation** for honest score reporting (Section 13)

Each of these choices was made deliberately. Understanding why each choice was made — and what the alternatives are — is the foundation for building a competitive submission.